# Online News Popularity — Macro F1 (A–E)
**Kaggle setup (do this first):**
1. Right sidebar → **Session options → Accelerator → GPU T4 x2** (or P100). Turn **Internet** off/on as you like (not needed).
2. Add the competition data via **Add Input**, or upload your zip as a dataset. The notebook auto-finds `train.csv`/`test.csv`.
3. **Run All.** Output: `/kaggle/working/submission.csv`.

**Approach:** feature engineering → 5-fold stratified CV → XGBoost (GPU) + CatBoost (GPU) + LightGBM (CPU) → blend OOF probabilities → **per-class probability weights tuned on OOF to maximise Macro F1** (this is the biggest win for Macro F1 with imbalanced classes).

No external data or pretrained models are used (competition rules).

In [1]:
import os, glob, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5
FAST = False          # True = quick smoke test (few trees); keep False for the real run
USE_GPU = True        # set False if no GPU accelerator is enabled
N_SEEDS = 1           # raise to 2-3 for a small extra boost if time allows

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'./**/{name}', recursive=True)
    assert hits, f'{name} not found - add the data via "Add Input"'
    return hits[0]

train = pd.read_csv(find('train.csv'), na_values=['NA'])
test  = pd.read_csv(find('test.csv'),  na_values=['NA'])
sub   = pd.read_csv(find('sample_submission.csv'))
print(train.shape, test.shape)
print(train.target_popularity.value_counts(normalize=True).sort_index())

(37144, 61) (2500, 60)
target_popularity
A    0.046172
B    0.106208
C    0.194648
D    0.363639
E    0.289333
Name: proportion, dtype: float64


## Feature engineering
Tree models handle `NA` natively, so we keep NaNs and just add a missing-count feature, ratios and combined categorical-ish features.

In [ ]:
def add_features(df):
    d = df.copy()
    eps = 1e-6
    d['n_missing'] = d.isna().sum(axis=1)

    # channel / weekday / LDA collapsed
    ch = [c for c in d.columns if c.startswith('data_channel_is_')]
    d['channel_id'] = np.where(d[ch].fillna(0).sum(axis=1) == 0, -1, d[ch].fillna(0).values.argmax(axis=1))
    wd = [c for c in d.columns if c.startswith('weekday_is_')]
    d['weekday_id'] = d[wd].fillna(0).values.argmax(axis=1)
    lda = [f'LDA_0{i}' for i in range(5)]
    d['lda_argmax'] = d[lda].fillna(0).values.argmax(axis=1)
    d['lda_max'] = d[lda].max(axis=1)
    d['lda_entropy'] = -(d[lda].clip(eps, 1) * np.log(d[lda].clip(eps, 1))).sum(axis=1)

    # content ratios / logs
    d['log_tokens'] = np.log1p(d['n_tokens_content'])
    d['links_per_token'] = d['num_hrefs'] / (d['n_tokens_content'] + 1)
    d['self_link_ratio'] = d['num_self_hrefs'] / (d['num_hrefs'] + 1)
    d['media_total'] = d['num_imgs'] + d['num_videos']
    d['imgs_per_token'] = d['num_imgs'] / (d['n_tokens_content'] + 1)
    d['title_content_ratio'] = d['n_tokens_title'] / (d['n_tokens_content'] + 1)
    d['kw_per_title'] = d['num_keywords'] / (d['n_tokens_title'] + 1)

    # keyword stats
    d['kw_range_avg'] = d['kw_max_avg'] - d['kw_min_avg']
    d['kw_avg_x_max'] = d['kw_avg_avg'] * d['kw_max_avg']
    d['kw_avg_ratio'] = d['kw_avg_avg'] / (d['kw_max_avg'] + 1)
    d['kw_max_min_ratio'] = d['kw_max_min'] / (d['kw_avg_min'] + 1)
    for c in ['kw_avg_avg','kw_max_avg','kw_min_avg','kw_avg_max','kw_min_max','kw_max_min','kw_avg_min']:
        d[f'log_{c}'] = np.log1p(d[c].clip(lower=0))

    # self-reference
    d['selfref_range'] = d['self_reference_max_shares'] - d['self_reference_min_shares']
    d['log_selfref_avg'] = np.log1p(d['self_reference_avg_sharess'].clip(lower=0))
    d['selfref_x_links'] = d['log_selfref_avg'] * d['num_self_hrefs']

    # sentiment
    d['pos_neg_diff'] = d['global_rate_positive_words'] - d['global_rate_negative_words']
    d['pol_range_pos'] = d['max_positive_polarity'] - d['min_positive_polarity']
    d['pol_range_neg'] = d['max_negative_polarity'] - d['min_negative_polarity']
    d['title_sub_x_pol'] = d['title_subjectivity'] * d['title_sentiment_polarity']
    d['subj_x_sent'] = d['global_subjectivity'] * d['global_sentiment_polarity']

    # interaction with channel & time
    d['kw_avg_x_channel'] = d['kw_avg_avg'] * (d['channel_id'] + 2)
    d['timedelta_x_kw'] = d['timedelta'] * d['kw_avg_avg'] / 1000
    return d

classes = sorted(train.target_popularity.unique())       # ['A','B','C','D','E']
cmap = {c: i for i, c in enumerate(classes)}
y = train.target_popularity.map(cmap).values
X = add_features(train.drop(columns=['id', 'target_popularity']))
X_test = add_features(test.drop(columns=['id']))
X_test = X_test[X.columns]
print(X.shape, X_test.shape)

## Models (5-fold stratified CV, out-of-fold probabilities)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
K = len(classes)
# mild class balancing so rare classes (A, B) are learned; final weights are tuned later anyway
cw = len(y) / (K * np.bincount(y))
cw = cw ** 0.5
sw_all = cw[y]

def run_xgb(seed):
    oof = np.zeros((len(X), K)); pred = np.zeros((len(X_test), K))
    for k, (tr, va) in enumerate(folds):
        m = xgb.XGBClassifier(
            n_estimators=200 if FAST else 3000, learning_rate=0.03, max_depth=6, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.6, reg_lambda=3, reg_alpha=0.5,
            tree_method='hist', device='cuda' if USE_GPU else 'cpu',
            objective='multi:softprob', eval_metric='mlogloss',
            early_stopping_rounds=100, random_state=seed + k, n_jobs=4)
        m.fit(X.iloc[tr], y[tr], sample_weight=sw_all[tr], eval_set=[(X.iloc[va], y[va])],
              sample_weight_eval_set=[sw_all[va]], verbose=False)
        oof[va] = m.predict_proba(X.iloc[va]); pred += m.predict_proba(X_test) / N_FOLDS
        print(f'  xgb fold {k}: iters={m.best_iteration}  F1={f1_score(y[va], oof[va].argmax(1), average="macro"):.4f}')
    return oof, pred

def run_cat(seed):
    oof = np.zeros((len(X), K)); pred = np.zeros((len(X_test), K))
    for k, (tr, va) in enumerate(folds):
        m = CatBoostClassifier(
            iterations=300 if FAST else 4000, learning_rate=0.06, depth=7, l2_leaf_reg=5,
            loss_function='MultiClass', eval_metric='MultiClass',
            task_type='GPU' if USE_GPU else 'CPU', random_seed=seed + k,
            od_type='Iter', od_wait=150, verbose=False)
        m.fit(X.iloc[tr], y[tr], sample_weight=sw_all[tr], eval_set=(X.iloc[va], y[va]))
        oof[va] = m.predict_proba(X.iloc[va]); pred += m.predict_proba(X_test) / N_FOLDS
        print(f'  cat fold {k}: iters={m.get_best_iteration()}  F1={f1_score(y[va], oof[va].argmax(1), average="macro"):.4f}')
    return oof, pred

def run_lgb(seed):
    oof = np.zeros((len(X), K)); pred = np.zeros((len(X_test), K))
    for k, (tr, va) in enumerate(folds):
        m = lgb.LGBMClassifier(
            n_estimators=200 if FAST else 3000, learning_rate=0.03, num_leaves=48, min_child_samples=30,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=3, reg_alpha=0.5,
            objective='multiclass', random_state=seed + k, n_jobs=-1, verbose=-1)
        m.fit(X.iloc[tr], y[tr], sample_weight=sw_all[tr], eval_set=[(X.iloc[va], y[va])],
              eval_sample_weight=[sw_all[va]],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = m.predict_proba(X.iloc[va]); pred += m.predict_proba(X_test) / N_FOLDS
        print(f'  lgb fold {k}: iters={m.best_iteration_}  F1={f1_score(y[va], oof[va].argmax(1), average="macro"):.4f}')
    return oof, pred

results = {}
for name, fn in [('xgb', run_xgb), ('cat', run_cat), ('lgb', run_lgb)]:
    t0 = time.time(); print(f'== {name}')
    oofs, preds = [], []
    for s in range(N_SEEDS):
        o, p = fn(SEED + 100 * s); oofs.append(o); preds.append(p)
    results[name] = (np.mean(oofs, 0), np.mean(preds, 0))
    print(f'{name}: OOF macro-F1 (argmax) = {f1_score(y, results[name][0].argmax(1), average="macro"):.4f}  ({time.time()-t0:.0f}s)')

## Blend + class-weight tuning for Macro F1
Predict `argmax(prob × w)` where the per-class weights `w` are tuned on the out-of-fold predictions via coordinate search.

In [ ]:
names = list(results)
# search blend weights on a coarse grid
best = (-1, None)
grid = np.arange(0, 1.01, 0.1)
for a in grid:
    for b in grid:
        if a + b > 1: continue
        w = dict(zip(names, [a, b, 1 - a - b]))
        p = sum(w[n] * results[n][0] for n in names)
        s = f1_score(y, p.argmax(1), average='macro')
        if s > best[0]: best = (s, w)
print('best blend (argmax F1):', round(best[0], 4), best[1])
blend_w = best[1]
oof_blend  = sum(blend_w[n] * results[n][0] for n in names)
test_blend = sum(blend_w[n] * results[n][1] for n in names)

def tune_class_weights(P, y, rounds=4):
    w = np.ones(P.shape[1]); cur = f1_score(y, (P * w).argmax(1), average='macro')
    for _ in range(rounds):
        for c in range(P.shape[1]):
            for v in np.arange(0.4, 3.01, 0.05):
                w2 = w.copy(); w2[c] = v
                s = f1_score(y, (P * w2).argmax(1), average='macro')
                if s > cur: cur, w = s, w2
    return w, cur

cwts, tuned = tune_class_weights(oof_blend, y)
print('class weights:', dict(zip(classes, cwts.round(2))))
print(f'OOF Macro-F1: argmax={f1_score(y, oof_blend.argmax(1), average="macro"):.4f} -> tuned={tuned:.4f}')

# honest check: tune weights on half the OOF rows, evaluate on the other half
idx = np.random.RandomState(0).permutation(len(y)); h = len(y) // 2
w_h, _ = tune_class_weights(oof_blend[idx[:h]], y[idx[:h]])
print('holdout-half F1 with weights tuned on other half:',
      round(f1_score(y[idx[h:]], (oof_blend[idx[h:]] * w_h).argmax(1), average='macro'), 4),
      '| untuned:', round(f1_score(y[idx[h:]], oof_blend[idx[h:]].argmax(1), average='macro'), 4))

## Write submission

In [ ]:
final = (test_blend * cwts).argmax(1)
sub = pd.DataFrame({'id': test['id'], 'target_popularity': [classes[i] for i in final]})
sub.to_csv('submission.csv', index=False)
print(sub.shape, sub.id.nunique() == len(test))
print(sub.target_popularity.value_counts(normalize=True).sort_index())
print('train dist:'); print(train.target_popularity.value_counts(normalize=True).sort_index())
sub.head()